## Smoothing of Athens AMS PM0.1 size distribution data

In [1]:
#Install packages
%pip install pandas matplotlib scipy openpyxl python-pptx

Note: you may need to restart the kernel to use updated packages.


In [2]:
#Install libraries
import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
from pptx import Presentation
from pptx.util import Inches
from scipy.signal import savgol_filter
from scipy.interpolate import UnivariateSpline

In [6]:
#Define working directories
wdir = Path("G:/Other computers/G-Box/Documents/Athens2025/AMS/ForCE/python_smoothing")
os.chdir(wdir) # Change directory
output_dir = r"G:\Other computers\G-Box\Documents\Athens2025\AMS\ForCE\python_smoothing\output" # Define the name or path of your output folder
os.makedirs(output_dir, exist_ok=True) # exist_ok=True prevents an error if you run this cell multiple times

In [7]:
#Import data
df_org = pd.read_excel(r"G:/Other computers/G-Box/Documents/Athens2025/AMS/ForCE/ForCE_all.xlsx", sheet_name="Org")
df_sulf = pd.read_excel(r"G:/Other computers/G-Box/Documents/Athens2025/AMS/ForCE/ForCE_all.xlsx", sheet_name="Sulf")
df_amm = pd.read_excel(r"G:/Other computers/G-Box/Documents/Athens2025/AMS/ForCE/ForCE_all.xlsx", sheet_name="Amm")
df_nitr = pd.read_excel(r"G:/Other computers/G-Box/Documents/Athens2025/AMS/ForCE/ForCE_all.xlsx", sheet_name="Nitr")
df_dva = pd.read_excel(r"G:/Other computers/G-Box/Documents/Athens2025/AMS/ForCE/ForCE_all.xlsx", sheet_name="diameter")

In [5]:
df_org.head()

,date,Dp1,Dp2,Dp3,Dp4,Dp5,Dp6,Dp7,Dp8,Dp9,...,Dp245,Dp246,Dp247,Dp248,Dp249,Dp250,Dp251,Dp252,Dp253,Dp254
0,2025-01-21 09:45:20.313,-4.153864,-0.078914,-1.766135,-2.162915,1.774498,-0.868704,-1.121998,1.773563,-0.132933,...,4.055248,0.598298,-0.554578,1.352855,2.014999,-4.449223,4.317322,11.791379,2.675941,-4.542269
1,2025-01-21 09:46:41.449,1.923562,0.302859,3.525627,2.500976,1.296383,-0.058434,-6.345384,-0.615081,0.224160,...,1.408002,-6.666773,3.048451,-0.259119,2.600710,-0.832199,1.470302,5.061443,-0.463963,3.449296
2,2025-01-21 09:48:00.285,-2.075545,-1.950254,0.710370,-0.215429,-2.151896,0.941146,-0.570199,-2.847325,-1.307005,...,0.563248,0.503948,-1.689811,-2.675272,-1.851079,1.847342,4.347182,5.072730,0.619947,2.587975
3,2025-01-21 09:49:20.402,-1.359560,-0.102572,-1.013979,-2.528446,3.414191,-1.466131,0.879087,-2.500315,-0.620146,...,1.331963,1.949073,-1.258520,-0.923306,0.381406,0.480577,-3.579501,0.937998,1.158535,-0.037535
4,2025-01-21 13:56:01.270,-1.127528,1.661226,-1.371443,0.160457,-1.392973,-1.379559,-0.782464,-1.103204,-1.029422,...,1.777912,1.838069,-1.061782,2.408873,1.712331,1.125528,1.697782,-3.280791,-0.134540,0.143877


In [8]:
# 1. Group your dataframes into a dictionary so we can loop through them easily
dfs = {
    'Org': df_org,
    'Sulf': df_sulf,
    'Amm': df_amm,
    'Nitr': df_nitr,
}

# Define your cutoff date and time
cutoff_date = pd.to_datetime('2025-01-24 06:00:00')

# 2. Loop through and clean each dataframe
# 3. Clean and filter (Bulletproof version)
for name, df in dfs.items():

    # Grab the exact name of the first column (index 0) and rename it
    first_col_name = df.columns[0]
    df.rename(columns={first_col_name: 'Datetime'}, inplace=True)

    # Convert to datetime
    df['Datetime'] = pd.to_datetime(df['Datetime'])

    # Filter out anything before Jan 24, 2025
    df_filtered = df[df['Datetime'] >= cutoff_date]

    # Set index
    df_filtered.set_index('Datetime', inplace=True)

    # Update dictionary
    dfs[name] = df_filtered

# 3. Reassign the cleaned dataframes back to your original variables
df_org = dfs['Org']
df_sulf = dfs['Sulf']
df_amm = dfs['Amm']
df_nitr = dfs['Nitr']

# Check your work!
print(f"df_org now starts on: {df_org.index.min()}")

df_org now starts on: 2025-01-24 06:00:07.604000


In [6]:
df_org.head()

,Dp1,Dp2,Dp3,Dp4,Dp5,Dp6,Dp7,Dp8,Dp9,Dp10,...,Dp245,Dp246,Dp247,Dp248,Dp249,Dp250,Dp251,Dp252,Dp253,Dp254
Datetime,,,,,,,,,,,,,,,,,,,,,
2025-01-24 06:00:07.604,-0.405190,0.666106,0.293778,-0.099074,-1.206704,-0.131344,-0.983358,-1.311486,-1.387151,-0.779629,...,-0.628233,-0.147963,-0.461790,-0.151834,0.051354,-0.189381,0.007967,0.137099,-0.171057,-0.402246
2025-01-24 06:03:06.387,-0.152407,-0.119628,-0.208312,-0.174069,-0.133087,-0.692358,-0.665138,-1.005118,-1.702555,-1.261072,...,-0.882426,-0.695428,1.293836,-0.840098,-1.093077,0.519710,-0.038059,0.768159,0.477501,0.601945
2025-01-24 06:06:09.453,0.876326,-0.176766,-0.158837,-0.712190,-0.730121,-1.186106,-1.299658,-0.677823,-0.956954,-1.250144,...,-0.584901,0.052698,0.081498,-0.492406,-0.595364,1.025338,0.841532,0.374486,-0.803640,-0.063787
2025-01-24 06:09:09.410,-0.315677,-0.553885,-0.252192,-0.377015,-0.832293,-0.514347,-1.012893,-0.803721,-1.366222,-0.766523,...,-0.042985,0.453506,0.354703,-1.016508,-0.165916,1.056287,-0.180384,0.231942,-0.004996,-1.113553
2025-01-24 06:12:09.652,0.478846,-0.039098,-0.869448,-0.332846,-0.584349,-0.919984,-1.582580,-1.160889,-1.162330,-1.232259,...,0.351723,-0.160903,0.111896,0.706007,-1.373279,1.174942,-0.501167,-0.708858,-0.681115,0.249158


In [7]:
# Create new dataframes for all the species that do have the negative values replaced with zero, setting a "floor" of 0 for all numerical values
# The .clip() method automatically returns a new DataFrame rather than modifying the old one

df_org_pos = df_org.clip(lower=0)
df_sulf_pos = df_sulf.clip(lower=0)
df_amm_pos = df_amm.clip(lower=0)
df_nitr_pos = df_nitr.clip(lower=0)

# Quick check to verify it worked
print(f"Minimum value in old Org dataframe: {df_org.min().min()}")
print(f"Minimum value in new Org dataframe: {df_org_pos.min().min()}")

Minimum value in old Org dataframe: -34.890472
Minimum value in new Org dataframe: 0.0


In [ ]:
# 1. Ensure we have our 1D list of physical diameters
diameters = df_dva.values.flatten()

# 2. Identify which columns correspond to diameters > 500 nm
# This creates a list of the specific column names (e.g., 'Dp150', 'Dp151'...)
cols_to_zero = df_org_pos.columns[diameters > 300]

# 3. Force all values in those specific columns to exactly 0 for all positive dataframes
df_org_pos.loc[:, cols_to_zero] = 0
df_sulf_pos.loc[:, cols_to_zero] = 0
df_amm_pos.loc[:, cols_to_zero] = 0
df_nitr_pos.loc[:, cols_to_zero] = 0
# 3. Force all values in those specific columns to exactly 0 for all negative dataframes
df_org.loc[:, cols_to_zero] = 0
df_sulf.loc[:, cols_to_zero] = 0
df_amm.loc[:, cols_to_zero] = 0
df_nitr.loc[:, cols_to_zero] = 0

# Quick check to verify how many bins were zeroed out
print(f"Total number of size bins: {len(diameters)}")
print(f"Number of size bins > 300nm forced to zero: {len(cols_to_zero)}")

In [ ]:
df_org.head()

In [18]:
df_ams_conc = pd.read_excel(r"G:/Other computers/G-Box/Documents/Athens2025/AMS/ForCE/ForCE_all_dilution_corrected.xlsx", sheet_name="species")
df_smps_vol = pd.read_excel(r"G:/Other computers/G-Box/Documents/Athens2025/AMS/ForCE/ForCE_all_dilution_corrected.xlsx", sheet_name="SMPS")
df_pm01 = pd.read_excel(r"G:/Other computers/G-Box/Documents/Athens2025/AMS/ForCE/ForCE_all_dilution_corrected.xlsx", sheet_name="pm01")
df_pm01_bc = pd.read_excel(r"G:/Other computers/G-Box/Documents/Athens2025/AMS/ForCE/ForCE_all_dilution_corrected.xlsx", sheet_name="bc")

In [16]:
df_ams_conc.head()

,date,organics,amm,sulfate,nitrate,chloride
0,2025-01-24 00:00:00,0.564227,0.001495,0.024527,0.020847,0.007670
1,2025-01-24 00:03:00,0.523595,0.002198,0.025687,0.022315,0.011045
2,2025-01-24 00:06:00,0.536824,0.003129,0.026888,0.020314,0.007969
3,2025-01-24 00:09:00,0.541293,0.002275,0.022828,0.019363,0.010658
4,2025-01-24 00:12:00,0.495001,0.001789,0.025781,0.021707,0.007670


In [19]:
df_ams_conc_12h_avg = df_ams_conc.resample('12h', offset='6h').mean()
df_smps_vol_12h_avg = df_smps_vol.resample('12h', offset='6h').mean()
df_pm01_12h_avg = df_pm01.resample('12h', offset='6h').mean()
df_pm01_bc_12h_avg = df_pm01_bc.resample('12h', offset='6h').mean()

df_ams_conc_12h_avg.head()

TypeError: Only valid with DatetimeIndex, TimedeltaIndex or PeriodIndex, but got an instance of 'RangeIndex'

In [9]:
# ==========================================
# 1. SETUP PARAMETERS & DIAMETERS
# ==========================================
window_size = 11
poly_order = 3

# Dilution correction factor (Total Flow / Sample Flow)
dilution_factor = 4 / 1.5

# Extract the physical diameters from df_dva by flattening its values
diameters = df_dva.values.flatten()

# Quick safety check to ensure sizes match before plotting
if len(diameters) != len(df_org.columns):
    raise ValueError(f"Mismatch! Found {len(diameters)} diameters but {len(df_org.columns)} size bins.")

# Group the dataframes
species_dfs = {
    'Org': df_org,
    'SO4': df_sulf,
    'NH4': df_amm,
    'NO3': df_nitr
}

averaged_dfs = {}

# ==========================================
# 2. APPLY SMOOTHING, DILUTION, & AVERAGING
# ==========================================
print("Smoothing data, applying dilution correction, and averaging to 6-hour intervals...")

for species, df in species_dfs.items():
    # Replace infinities with NaNs, then fill all NaNs with 0
    df_clean = df.replace([np.inf, -np.inf], np.nan).fillna(0)

    # 1. Apply Savitzky-Golay filter across the columns (keeping all negatives!)
    smoothed_array = savgol_filter(df_clean.values, window_length=window_size, polyorder=poly_order, axis=1)
    df_smoothed = pd.DataFrame(smoothed_array, index=df_clean.index, columns=df_clean.columns)

    # 2. Apply the dilution correction
    df_smoothed = df_smoothed * dilution_factor

    # 3. Resample to 4-hour averages (Let the positive and negative noise cancel out here!)
    df_12h_avg = df_smoothed.resample('12h', offset='6h').mean()

    # 4. NOW clip the final averaged data to 0
    averaged_dfs[species] = df_12h_avg.clip(lower=0)

# ==========================================
# 3. GENERATE POWERPOINT PRESENTATION
# ==========================================
print("Generating PowerPoint slides...")

prs = Presentation()
prs.slide_width = Inches(13.333) # 16:9 widescreen
prs.slide_height = Inches(7.5)

timestamps = averaged_dfs['Org'].index

for timestamp in timestamps:
    # Skip plotting if this specific hour chunk is completely empty
    if averaged_dfs['Org'].loc[timestamp].isna().all():
        continue

    fig, ax = plt.subplots(figsize=(12, 7))

    # Plot each species
    ax.plot(diameters, averaged_dfs['NH4'].loc[timestamp], label='NH4', color='#FFA500', linewidth=2.5)
    ax.plot(diameters, averaged_dfs['Org'].loc[timestamp], label='Org', color='#228B22', linewidth=2.5)
    ax.plot(diameters, averaged_dfs['SO4'].loc[timestamp], label='SO4', color='#FF2400', linewidth=2.5)
    ax.plot(diameters, averaged_dfs['NO3'].loc[timestamp], label='NO3', color='#87CEEB', linewidth=2.5)

    # Formatting X-axis (Log scale for particle diameters)
    ax.set_xscale('log')
    ax.set_xlim([10, 2000])

    # Formatting Labels
    ax.set_xlabel('Particle Diameter (nm)', fontsize=14)
    ax.set_ylabel('dM/dlogDva (µg m⁻³)', fontsize=14)

    # Formatting Title
    formatted_time = timestamp.strftime('%Y-%m-%d %H:%M')
    ax.set_title(f'AMS size distributions (CE=1) – {formatted_time}', fontsize=16, loc='left', pad=40)

    # Formatting Legend
    ax.legend(title='Species', loc='upper center', bbox_to_anchor=(0.5, 1.10),
              ncol=4, frameon=False, fontsize=12, title_fontsize=14)

    # Formatting Grid
    ax.grid(True, which="major", ls="-", alpha=0.3)
    ax.grid(True, which="minor", ls="-", alpha=0.1)

    # Save and insert image
    temp_img = "temp_slide_plot.png"
    fig.savefig(temp_img, bbox_inches='tight', dpi=150)
    plt.close(fig)

    slide = prs.slides.add_slide(prs.slide_layouts[6])
    slide.shapes.add_picture(temp_img, Inches(0.5), Inches(0.5), width=Inches(10))

# ==========================================
# 4. SAVE TO SPECIFIC DIRECTORY
# ==========================================
output_dir = r"G:\Other computers\G-Box\Documents\Athens2025\AMS\ForCE\python_smoothing\output"
os.makedirs(output_dir, exist_ok=True)

output_filepath = os.path.join(output_dir, "AMS_Smoothed_12Hour_Averages_neg_DilutionCorrected.pptx")

# Save the presentation directly to that path
prs.save(output_filepath)

# Clean up the temporary image
if os.path.exists("temp_slide_plot.png"):
    os.remove("temp_slide_plot.png")

print(f"The PowerPoint is saved here:\n{output_filepath}")

Smoothing data, applying dilution correction, and averaging to 6-hour intervals...
Generating PowerPoint slides...
The PowerPoint is saved here:
G:\Other computers\G-Box\Documents\Athens2025\AMS\ForCE\python_smoothing\output\AMS_Smoothed_12Hour_Averages_neg_DilutionCorrected.pptx


In [13]:
# ==========================================
# 5. EXPORT DATA TO EXCEL
# ==========================================
print("Exporting data to Excel...")
excel_filepath = os.path.join(output_dir, "AMS_Smoothed_12Hour_Averages_neg_DilutionCorrected.xlsx")

# Use pandas ExcelWriter to create multiple sheets in one file
with pd.ExcelWriter(excel_filepath) as writer:
    # Save the physical diameters to the first sheet so you have the X-axis reference
    pd.DataFrame({'Diameter_nm': diameters}).to_excel(writer, sheet_name='Diameters', index=False)

    # Loop through the smoothed & averaged dataframes and save each to its own sheet
    for species, df in averaged_dfs.items():
        df.to_excel(writer, sheet_name=species)

    # Save the additional 12-hour averaged dataframes to their own sheets
    df_ams_conc_12h_avg.to_excel(writer, sheet_name='AMS_Conc')
    df_smps_vol_12h_avg.to_excel(writer, sheet_name='SMPS_Vol')
    df_pm01_12h_avg.to_excel(writer, sheet_name='PM01')
    df_pm01_bc_12h_avg.to_excel(writer, sheet_name='PM01_BC')

print(f"The Excel data is saved here:\n{excel_filepath}")

Exporting data to Excel...
The Excel data is saved here:
G:\Other computers\G-Box\Documents\Athens2025\AMS\ForCE\python_smoothing\output\AMS_Smoothed_12Hour_Averages_neg_DilutionCorrected.xlsx
